In [1]:
import torch
import time
import pandas as pd
import os

from torchvision.models import mobilenet_v2

In [2]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu126
CUDA available: True


In [3]:
!pip install torch-pruning

In [4]:
model = mobilenet_v2(weights="DEFAULT")
model.eval()

print("Model loaded.")

Model loaded.


In [5]:
import torch_pruning as tp

In [7]:
example_inputs = torch.randn(1, 3, 224, 224)

In [8]:
importance = tp.importance.MagnitudeImportance(p=2)

In [9]:
example_inputs = torch.randn(1, 3, 224, 224)

ignored_layers = [model.classifier]

pruner = tp.pruner.MagnitudePruner(
    model,
    example_inputs,
    importance=importance,
    pruning_ratio=0.2,
    ignored_layers=ignored_layers
)

In [10]:
pruner.step()

print("Structural pruning applied.")

Structural pruning applied.


In [11]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters after structural pruning: {total_params:,}")

Total parameters after structural pruning: 2,460,140


In [12]:
torch.save(model.state_dict(), "../models/phase2_structural_pruned_model.pth")

size_mb = os.path.getsize("../models/phase2_structural_pruned_model.pth") / (1024 * 1024)

print(f"Structural pruned model size: {size_mb:.2f} MB")

Structural pruned model size: 9.59 MB


In [13]:
def benchmark(model, device, input_tensor, runs=100):

    model = model.to(device)
    input_tensor = input_tensor.to(device)

    # Warm-up
    for _ in range(10):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    for _ in range(runs):
        with torch.no_grad():
            _ = model(input_tensor)

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    return (end - start) / runs

In [15]:
input_tensor = torch.randn(1, 3, 224, 224)

cpu_latency = benchmark(model, "cpu", input_tensor)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.053269 seconds


PHASE 2

In [16]:
quantized_structural_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

print("Structural model quantized.")

Structural model quantized.


C:\Users\user\AppData\Local\Temp\ipykernel_18528\2344469378.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_structural_model = torch.quantization.quantize_dynamic(


In [17]:
torch.save(
    quantized_structural_model.state_dict(),
    "../models/structural_prune_then_quantized_model.pth"
)

In [18]:
size_mb = os.path.getsize(
    "../models/structural_prune_then_quantized_model.pth"
) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 6.67 MB


In [19]:
input_tensor = torch.randn(1, 3, 224, 224)

cpu_latency = benchmark(
    quantized_structural_model,
    "cpu",
    input_tensor
)

print(f"CPU Average Latency: {cpu_latency:.6f} seconds")

CPU Average Latency: 0.054325 seconds
